[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kithhooni-commits/ds-practice/blob/main/%EC%8B%A4%EC%8A%B55/colab_day3.ipynb)

# 실습5 Day 3 — Denoising + Deconvolution

```
Day 1   g = f + n          입력 24.67 dB   노이즈만
Day 2   g = h * f          입력  7.89 dB   흐림만
Day 3   g = h * f + n      입력  8.02 dB   둘 다
```

**3일차 노이즈는 1일차와 파일별로 완전히 동일하다** — `test_deconv_noise/noise_meta.json` 의
종류·σ 가 `test_noise_only` 와 100/100 일치한다. dipole 도 2일차와 같다.
지금까지 만든 것이 그대로 합쳐진다.

## 2일차 답이 무너진다

| | Day 2 | Day 3 |
|---|---|---|
| Wiener K→0 | **109.86** | **−24.24** |
| Wiener 최적 K | — | 14.59 (K=3e-2) |
| 배포 baseline | U-Net 25.59 | **U-Net 25.01** |

역산이 노이즈를 32 dB 증폭한다. K 를 키워 막으면 이번엔 정보를 버려서 14.59 에 그친다.
**고전 기법 단독으로는 딥러닝을 못 이긴다** — 1·2일차와 정반대 상황이다.

## 그래서 나눠서 푼다

```
x₀ = Wiener(g, K)                    2일차 — 흐림을 되돌린다
반복:  z = 디노이저(x)                1일차 — 노이즈를 지운다
      x = (D·G + λZ)/(D² + λ)        물리 제약, 닫힌 해
```

한 번에 다 하려면 K 를 크게 잡아 정보를 버려야 하지만, 나눠서 반복하면 둘 다 살린다.

## 0. 런타임 · Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 1. 데이터 준비

**3일차용으로 갱신된 dataset** 이 필요하다 (`test_deconv_noise` 포함).
`code_denoising+deconv` zip 에는 배포 베이스라인 체크포인트(13.43M)가 들어 있다.

In [ ]:
import os, zipfile
from pathlib import Path

SEARCH_ROOT = Path("/content/drive/MyDrive")
WORK = Path("/content/data")
DATA_ROOT = None
WANT = ("dataset", "code_denoising+deconv")

WORK.mkdir(parents=True, exist_ok=True)
for z in sorted(p for d in ("*.zip", "*/*.zip") for p in SEARCH_ROOT.glob(d)):
    tag = next((w for w in WANT if z.name.startswith(w)), None)
    if tag is None:
        continue
    print("푸는 중:", z.name)
    with zipfile.ZipFile(z) as f:
        f.extractall(WORK)

def looks_like(p):
    return (p / "train").is_dir() and (p / "test_deconv_noise").is_dir()

if DATA_ROOT is None:
    seen = [WORK] + [p for d in ('*', '*/*') for p in WORK.glob(d) if p.is_dir()]
    seen += [SEARCH_ROOT] + [p for d in ('*', '*/*') for p in SEARCH_ROOT.glob(d) if p.is_dir()]
    cands = [p for p in seen if looks_like(p)]
    if not cands:
        print("WORK 안:", [x.name for x in WORK.iterdir()])
        raise SystemExit("test_deconv_noise 가 있는 dataset 을 못 찾았다. 3일차 zip 인지 확인할 것")
    DATA_ROOT = cands[0]

DATA_ROOT = Path(DATA_ROOT)
os.environ["DS_DATA"] = str(DATA_ROOT)
print("\nDATA_ROOT =", DATA_ROOT)
for sub in ("train", "val", "test_label", "test_deconv_noise"):
    q = DATA_ROOT / sub
    n = len(list(q.glob("**/*.npy"))) if q.exists() else 0
    print(f"{'OK  ' if n else '없음'} {sub:<20} {n:>5} npy")

## 2. 코드 받기

In [ ]:
REPO = Path("/content/ds-practice")
if REPO.exists():
    !cd "{REPO}" && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/kithhooni-commits/ds-practice.git "{REPO}"

SRC = REPO / "실습5" / "src" / "deconv"
RUNS = Path("/content/runs")
RUNS.mkdir(exist_ok=True)
!cd "{REPO}" && git log --oneline -1

## 3. 넘어야 할 선 — 배포 베이스라인과 고전 기법

배포 체크포인트는 End2End U-Net (chans 64, 4 pool, 13.43M, 100 epoch, L2) 이다.
우리 `models.py` 의 `Unet(features=64)` 과 구조가 같아 그대로 불러온다.

In [ ]:
BASE = Path("/content/data/code_denoising+deconv/checkpoint_baseline_best.ckpt")
if not BASE.exists():
    BASE = next(Path("/content/data").rglob("checkpoint_baseline_best.ckpt"))
print("baseline:", BASE)
!cd "{SRC}" && python eval_day3.py --data "{DATA_ROOT}" --ckpt "{BASE}" --wiener

## 4. 학습

`--noise-model challenge` 가 핵심이다. 1일차 4종 노이즈를 **흐림 뒤에** 얹어
`test_deconv_noise` 와 같은 조건을 만든다. 크롭은 하지 않는다 — deconvolution 은
전역 연산이라 조각만 보면 복원에 필요한 정보가 조각 밖에 있다.

| 모델 | 아이디어 | 2일차 성적 |
|---|---|---|
| `unet` | 배포 방식. end-to-end | 28.59 |
| `dcnet` | 신뢰 대역은 G/D 로 못 박고 null cone 만 학습 | 42.93 |
| `unrolled` | 데이터 정합 ↔ 디노이저를 N번 반복 | — |

3일차는 노이즈가 있으므로 `tau` 를 2일차보다 크게 잡는다. 작게 잡으면 1/D 가
노이즈를 증폭한다.

### ① DC-Net — 2일차 최고 구조를 3일차 조건으로

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model dcnet --refine unet --features 32 --tau 0.15 \
    --noise-model challenge \
    --epochs 40 --batch 16 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag dcnet_d3

### ② 전개형 — 데이터 정합과 디노이저를 5번 반복

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model unrolled --refine unet --features 32 --unroll-iters 5 \
    --noise-model challenge \
    --epochs 40 --batch 8 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag unrolled_d3

### ③ 대조군 — 배포와 같은 end-to-end U-Net (크롭 없이, 우리 레시피로)

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model unet --features 64 \
    --noise-model challenge \
    --epochs 60 --batch 8 --lr 2e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag unet_d3

## 5. 평가 — 제출값 산출

`test_deconv_noise` 100장을 `test_label` 로 채점한다. 배포 지표 구현 그대로.

In [ ]:
import glob, json
from pathlib import Path

for ck in sorted(RUNS.glob("*/checkpoints/checkpoint_best.ckpt")):
    print(f"\n{'='*64}\n  {ck.parent.parent.name}\n{'='*64}")
    !cd "{SRC}" && python eval_day3.py --data "{DATA_ROOT}" --ckpt "{ck}"

### 결과 요약 + Drive 저장

In [ ]:
import shutil, json, glob
from pathlib import Path

OUT = Path("/content/drive/MyDrive/실습프로젝트/runs_day3")
OUT.mkdir(parents=True, exist_ok=True)
for r in Path("/content/runs").glob("*"):
    shutil.copytree(r, OUT / r.name, dirs_exist_ok=True)

print(f"\n{'run':<44}{'val PSNR':>10}{'SSIM':>9}")
print("-" * 63)
for h in sorted(glob.glob("/content/runs/*/history.json")):
    d = json.load(open(h))
    b = max(d, key=lambda x: x["val_psnr"])
    print(f'{Path(h).parent.name:<44}{b["val_psnr"]:>10.2f}{b["val_ssim"]:>9.4f}')

## 6. 제출

`challenge_score.xlsx` 에 **성함(팀명) · PSNR · SSIM · Supervised / Self-supervised / Others**
를 적는다. PSNR 은 소수 셋째자리, SSIM 은 넷째자리에서 반올림.

| | PSNR | SSIM | 분류 |
|---|---|---|---|
| 입력 (blur + noise) | 8.02 | −0.0187 | — |
| Wiener 최적 K=3e-2 | 14.59 | 0.4322 | Others |
| 배포 baseline (End2End U-Net) | 25.01 | 0.8149 | Supervised |
| 우리 모델 | ? | ? | Supervised |

## 7. 2차 시도 — 전개형이 ep37 에서 멈춘 뒤

1차 전개형(unet f32, blind)이 **val 25.91 에서 정체**했다. 에폭이 병목이 아니다.
숫자로 확인한 병목은 두 가지다.

- **σ 가 이미지마다 200배 차이난다** (0.0007 ~ 0.13). blind 모델 하나로 덮으려면
  평균에 타협해야 한다. σ<0.05 에서 23.91 dB, σ>=0.10 에서 17.64 dB — 6 dB 차이.
- **용량.** 1일차에서 DRUNet 이 DnCNN 을 2.9 dB 이겼다. 전개형 안에서도 같을 것이다.

**먼저 셀 6(코드 받기)을 다시 실행할 것.** 아래 플래그는 방금 추가된 것이라
`git pull` 없이는 `unrecognized arguments` 가 난다.


In [ ]:
# 1일차 DRUNet 체크포인트를 찾는다 (전개형의 사전지식 자리 초기값으로 쓴다)
# 없으면 --init-refine 없이 돌려도 된다. 있으면 훨씬 빨리 수렴한다.
import glob
from pathlib import Path

CAND = []
for pat in ("/content/drive/MyDrive/**/*drunet*.ckpt", "/content/drive/MyDrive/**/*.ckpt",
            "/content/runs/**/checkpoint_best.ckpt"):
    CAND += glob.glob(pat, recursive=True)

D1 = None
for c in sorted(set(CAND)):
    try:
        import torch
        ck = torch.load(c, map_location="cpu", weights_only=False)
    except Exception:
        continue
    if ck.get("model") == "drunet" and not ck.get("label_free"):
        print(f"  후보  {c}   val {ck.get('val_psnr', float('nan')):.2f} dB")
        if D1 is None or ck.get("val_psnr", 0) > best:
            D1, best = c, ck.get("val_psnr", 0)

INIT = f'--init-refine "{D1}"' if D1 else ""
print(f"\n선택: {D1 if D1 else '없음 — 무작위 초기화로 진행한다 (그래도 돌아간다)'}")

### ① 주력 — σ 조건화 + DRUNet + 1일차 가중치 warm start

`estimate_sigma` 가 **측정치만 보고** σ 를 읽는다. `|D|<0.02` 인 주파수엔 신호가
실려올 수 없으니 거기 남은 건 전부 노이즈다 (파세발). val 40장 상대오차 중앙값 1.9%.
정답도 `noise_meta.json` 도 쓰지 않는다.

warm start 는 σ 채널을 0 으로 두고 시작하므로 **첫 순간엔 1일차 디노이저와 정확히
같게** 동작하고, 거기서부터 σ 를 쓰는 법을 배운다 (차이 0.00e+00 로 확인).

`checkpoint_best` 는 매 에폭 갱신되니 **중간에 끊어도 쓸 수 있다.**

In [ ]:
!cd "{SRC}" && python train_deconv.py     --model unrolled --refine drunet --features 48 --unroll-iters 4     --sigma-map {INIT}     --noise-model challenge --input measure     --epochs 60 --batch 4 --lr 2e-4 --loss charbonnier --clip-grad 1.0 --workers 8     --data "{DATA_ROOT}" --out "{RUNS}" --tag u_drunet_sig

### ② 대조군 — σ 조건화만 뺀다

①과 이 둘의 차이가 곧 **σ 조건화가 번 점수**다. 발표의 ablation 에 필요하다.
시간이 없으면 이건 건너뛰고 ①을 더 오래 돌려도 된다.

In [ ]:
!cd "{SRC}" && python train_deconv.py     --model unrolled --refine drunet --features 48 --unroll-iters 4     --noise-model challenge --input measure     --epochs 60 --batch 4 --lr 2e-4 --loss charbonnier --clip-grad 1.0 --workers 8     --data "{DATA_ROOT}" --out "{RUNS}" --tag u_drunet_blind

### ③ 방법 B 대조군 — 측정치 영역에서 노이즈만 지우고 Wiener 가 역산

배포 노트북의 방법 B. 노이즈가 흐림 **뒤에** 붙었으므로 측정치 위에서는 백색이고,
그게 디노이저가 가장 잘하는 조건이다. 평가 때 `--sweep-K` 로 K 를 val 에서 고른다.

In [ ]:
!cd "{SRC}" && python train_deconv.py     --model drunet --features 64 --target measure     --noise-model challenge --input measure     --epochs 60 --batch 8 --lr 2e-4 --loss charbonnier --clip-grad 1.0 --workers 8     --data "{DATA_ROOT}" --out "{RUNS}" --tag methodB

## 8. 평가 — 4x self-ensemble 포함

dipole 이 견디는 대칭만 쓴다: **좌우 뒤집기 · 상하 뒤집기 · 180도 회전** (오차 7e-16).
전치와 90도 회전은 B0 방향을 돌려버려 **연산자 자체가 바뀐다** (오차 3.17) — 1일차의
8x self-ensemble 을 그대로 가져오면 오히려 손해다.

K 는 전부 **val 에서** 고른다. test 는 채점에만 쓴다.

In [ ]:
for ck in sorted(RUNS.glob("*/checkpoints/checkpoint_best.ckpt")):
    name = ck.parent.parent.name
    if not any(t in name for t in ("u_drunet_sig", "u_drunet_blind", "methodB")):
        continue
    sweep = "--sweep-K" if "methodB" in name else ""
    print(f"\n{'='*64}\n  {name}\n{'='*64}")
    !cd "{SRC}" && python eval_day3.py --data "{DATA_ROOT}" --ckpt "{ck}" --self-ensemble {sweep}